# Upload input all-in-one ke akun Kaggle baru

Notebook Colab ini mengambil artefak dari satu folder Google Drive `Coffee_Bean_Detection`, lalu membuat atau memperbarui **satu private Kaggle Dataset** yang dipakai oleh notebook `Faruq_V3_Top_Controls_All_Sequential_Kaggle.ipynb`.

Dataset hasil upload berisi arsip Faruq-v3 utuh sebagai `.tar.bin` agar Kaggle tidak membongkar atau mengonversi label TXT, empat checkpoint sumber, evidence, manifest SHA, dan hasil FCT0 seed-123 yang dapat dipulihkan. Tidak ada training dan tidak ada akses test.

Sebelum menjalankan, buat dua Colab secrets dan aktifkan notebook access: `KAGGLE_USERNAME` dan `KAGGLE_API_TOKEN`. Setelah itu pilih **Runtime → Run all**.


In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive', force_remount=True)

import hashlib, json, os, shutil, subprocess, sys, tarfile, time, zipfile
from pathlib import Path, PurePosixPath

REL_ARCHIVE = Path('bundles/faruq-development-v3-grouped.tar')
REL_CONTROLS = Path('bundles/top-controls-multiseed-core-v1.zip')
PREFERRED = Path('/content/drive/MyDrive/Coffee_Bean_Detection')
SEARCH_ROOTS = [Path('/content/drive/MyDrive'), Path('/content/drive/.shortcut-targets-by-id')]

def find_project_artifact(relative: Path) -> Path:
    preferred = PREFERRED / relative
    if preferred.is_file():
        return preferred
    suffix = ('Coffee_Bean_Detection/' + relative.as_posix()).lower()
    matches = []
    for root in SEARCH_ROOTS:
        if root.exists():
            matches.extend(
                path for path in root.rglob(relative.name)
                if path.is_file() and path.as_posix().lower().endswith(suffix)
            )
    matches = sorted(set(matches), key=lambda path: (len(str(path)), str(path)))
    if not matches:
        raise FileNotFoundError(
            f'Artefak tidak ditemukan: Coffee_Bean_Detection/{relative.as_posix()}. '
            'Tambahkan shortcut folder proyek ke My Drive akun Colab ini.'
        )
    sizes = {path.stat().st_size for path in matches}
    if len(sizes) != 1:
        raise RuntimeError(f'Ada duplikat berbeda ukuran untuk {relative}: {matches}')
    if len(matches) > 1:
        print(f'{relative.name}: {len(matches)} alias Drive identik ukuran; memakai {matches[0]}')
    return matches[0]

ARCHIVE_SOURCE = find_project_artifact(REL_ARCHIVE)
CONTROLS_ZIP = find_project_artifact(REL_CONTROLS)
assert ARCHIVE_SOURCE.stat().st_size > 1_000_000_000, 'Arsip Faruq terlalu kecil/tidak utuh.'
assert CONTROLS_ZIP.stat().st_size > 10_000_000, 'Bundle top-controls terlalu kecil/tidak utuh.'
print('ARSIP FARUQ :', ARCHIVE_SOURCE, f'({ARCHIVE_SOURCE.stat().st_size / 1e9:.2f} GB)')
print('TOP CONTROLS:', CONTROLS_ZIP, f'({CONTROLS_ZIP.stat().st_size / 1e6:.1f} MB)')
print('PREFLIGHT DRIVE: PASS')


In [ ]:
STAGING = Path('/content/kaggle-upload/faruq-v3-top-controls-all-in-one-v1')
if STAGING.exists():
    shutil.rmtree(STAGING)
STAGING.mkdir(parents=True)

def copy_with_sha(source: Path, target: Path, label: str) -> str:
    digest = hashlib.sha256()
    copied = 0
    next_report = 256 * 1024 * 1024
    total = source.stat().st_size
    with source.open('rb') as src, target.open('wb') as dst:
        while True:
            block = src.read(16 * 1024 * 1024)
            if not block:
                break
            dst.write(block)
            digest.update(block)
            copied += len(block)
            if copied >= next_report:
                print(f'{label}: {copied / 1e9:.2f}/{total / 1e9:.2f} GB disalin', flush=True)
                next_report += 256 * 1024 * 1024
    shutil.copystat(source, target)
    if copied != total:
        raise RuntimeError(f'Copy tidak lengkap: {source}')
    return digest.hexdigest()

ARCHIVE_TARGET = STAGING / 'faruq-development-v3-grouped.tar.bin'
archive_sha = copy_with_sha(ARCHIVE_SOURCE, ARCHIVE_TARGET, 'FARUQ')

with zipfile.ZipFile(CONTROLS_ZIP) as bundle:
    for member in bundle.infolist():
        path = PurePosixPath(member.filename)
        if path.is_absolute() or '..' in path.parts:
            raise RuntimeError(f'Path ZIP tidak aman: {member.filename}')
    bundle.extractall(STAGING)

manifest_path = STAGING / 'top_controls_kaggle_manifest.json'
if not manifest_path.is_file():
    matches = list(STAGING.rglob('top_controls_kaggle_manifest.json'))
    if len(matches) != 1:
        raise FileNotFoundError(f'Manifest top-controls harus tepat satu; ditemukan {matches}')
    wrapper = matches[0].parent
    for child in list(wrapper.iterdir()):
        destination = STAGING / child.name
        if destination.exists():
            raise RuntimeError(f'Bentrok saat meratakan bundle: {destination}')
        shutil.move(str(child), str(destination))
    manifest_path = STAGING / 'top_controls_kaggle_manifest.json'

controls = json.loads(manifest_path.read_text(encoding='utf-8'))
assert controls['format'] == 'coffee_detector.top_controls.kaggle_manifest.v1'
assert controls['test_images_included'] is False
for name, contract in controls['artifacts'].items():
    path = STAGING / name
    assert path.is_file(), path
    digest = hashlib.sha256(path.read_bytes()).hexdigest()
    assert path.stat().st_size == int(contract['bytes']) and digest == contract['sha256'], name

with tarfile.open(ARCHIVE_TARGET, 'r') as archive:
    names = [member.name for member in archive.getmembers() if member.isfile()]
counts = {
    'train_images': sum('/train/images/' in name for name in names),
    'train_labels': sum('/train/labels/' in name and name.endswith('.txt') for name in names),
    'val_images': sum('/val/images/' in name for name in names),
    'val_labels': sum('/val/labels/' in name and name.endswith('.txt') for name in names),
}
expected = {'train_images': 1665, 'train_labels': 1665, 'val_images': 294, 'val_labels': 294}
if counts != expected:
    raise RuntimeError(f'Kontrak arsip Faruq gagal: {counts}, expected={expected}')
if any('/test/' in name for name in names):
    raise RuntimeError('STOP: arsip tidak boleh memuat test.')

upload_manifest = {
    'format': 'coffee_detector.top_controls.all_in_one_upload.v1',
    'faruq_archive': {'name': ARCHIVE_TARGET.name, 'bytes': ARCHIVE_TARGET.stat().st_size, 'sha256': archive_sha},
    'top_controls_manifest_sha256': hashlib.sha256(manifest_path.read_bytes()).hexdigest(),
    'dataset_contract': counts,
    'test_images_included': False,
}
(STAGING / 'all_in_one_upload_manifest.json').write_text(json.dumps(upload_manifest, indent=2) + '\n', encoding='utf-8')
print('STAGING VALID: PASS')
print('FARUQ SHA256:', archive_sha)
print('FILES:', len([path for path in STAGING.rglob('*') if path.is_file()]))


In [ ]:
username = userdata.get('KAGGLE_USERNAME')
token = userdata.get('KAGGLE_API_TOKEN')
assert username and token, 'Tambahkan dan aktifkan secret KAGGLE_USERNAME serta KAGGLE_API_TOKEN.'
username = username.strip()
assert username and '/' not in username, 'KAGGLE_USERNAME tidak valid.'
os.environ['KAGGLE_USERNAME'] = username
os.environ['KAGGLE_API_TOKEN'] = token
os.environ['KAGGLE_KEY'] = token

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'kaggle'], check=True)
DATASET_SLUG = 'faruq-v3-top-controls-all-in-one-v1'
dataset_id = f'{username}/{DATASET_SLUG}'
metadata = {
    'title': 'Faruq V3 Top Controls All In One V1',
    'id': dataset_id,
    'licenses': [{'name': 'other'}],
    'isPrivate': True,
}
(STAGING / 'dataset-metadata.json').write_text(json.dumps(metadata, indent=2) + '\n', encoding='utf-8')

common = ['-p', str(STAGING), '--keep-tabular', '--dir-mode', 'zip']
message = 'All-in-one Faruq-v3 archive and top-control multi-seed inputs with SHA contracts'
version = subprocess.run(
    ['kaggle', 'datasets', 'version', *common, '-m', message],
    text=True, capture_output=True
)
if version.returncode == 0:
    print(version.stdout)
    action = 'UPDATED'
else:
    combined = (version.stdout + '\n' + version.stderr).lower()
    missing = any(text in combined for text in ('not found', '404', 'does not exist'))
    if not missing:
        print(version.stdout)
        print(version.stderr)
        raise RuntimeError(f'Kaggle version gagal sebelum create: returncode={version.returncode}')
    print('Dataset belum ada pada akun ini; membuat private dataset baru...', flush=True)
    subprocess.run(['kaggle', 'datasets', 'create', *common], check=True)
    action = 'CREATED'

url = f'https://www.kaggle.com/datasets/{dataset_id}'
print('UPLOAD:', action)
print('DATASET:', dataset_id)
print('URL:', url)
print('PRIVATE: True')


In [ ]:
required = ('faruq-development-v3-grouped.tar.bin', 'top_controls_kaggle_manifest.json')
listing = ''
for attempt in range(12):
    check = subprocess.run(
        ['kaggle', 'datasets', 'files', dataset_id, '--page-size', '100', '--csv'],
        text=True, capture_output=True
    )
    listing = check.stdout
    if check.returncode == 0 and all(name in listing for name in required):
        break
    print(f'Menunggu indeks Kaggle: {attempt + 1}/12', flush=True)
    time.sleep(10)
else:
    print(listing)
    print(check.stderr)
    raise RuntimeError('Upload diterima tetapi dua file wajib belum muncul di indeks Kaggle.')

print('VERIFIKASI KAGGLE: PASS')
print('Pasang SATU dataset ini sebagai Input notebook training:', dataset_id)
print('Lalu aktifkan GPU + Internet dan gunakan Save Version → Save & Run All.')
